# Classificação do Desempenho Acadêmico de Estudantes utilizando SVM

## Introdução

A análise de dados educacionais pode contribuir para a compreensão de padrões relacionados ao desempenho acadêmico dos estudantes. Por meio de técnicas de aprendizado de máquina, é possível analisar diferentes características presentes em uma base de dados e investigar sua relação com determinados níveis de desempenho.

Neste projeto, será utilizado o conjunto de dados **Students Performance in Exams**, disponibilizado na plataforma Kaggle. A base contém informações sobre estudantes, incluindo gênero, grupo étnico, escolaridade dos pais, tipo de almoço, participação em curso preparatório e notas obtidas nas avaliações de Matemática, Leitura e Escrita.

A partir das três notas disponíveis, será calculada a média de cada estudante e criada uma classificação de desempenho dividida em três categorias: **Baixo, Médio e Alto**. Posteriormente, será utilizado o algoritmo **Support Vector Machine (SVM)** para investigar a possibilidade de classificar esses níveis de desempenho a partir das demais características disponíveis no conjunto de dados.

Durante o desenvolvimento serão realizadas etapas de carregamento, exploração, limpeza e preparação dos dados, transformação das variáveis categóricas, divisão dos dados em treinamento e teste e padronização das características. Em seguida, serão treinados modelos SVM utilizando os kernels **Linear e RBF**, além da realização de ajuste de hiperparâmetros.

O desempenho dos modelos será analisado por meio de métricas como **acurácia, precisão, recall e F1-score**, além da utilização de matrizes de confusão e representações gráficas. Dessa forma, o projeto busca não apenas aplicar o algoritmo SVM, mas também analisar suas possibilidades e limitações na classificação do desempenho acadêmico com base nas características disponíveis.

In [ ]:


import numpy as np 
import pandas as pd 

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub


## 1. Carregamento do Dataset

Nesta primeira etapa, será realizado o carregamento do conjunto de dados utilizado no projeto. O dataset **Students Performance in Exams**, disponibilizado na plataforma Kaggle, contém informações relacionadas ao desempenho acadêmico de estudantes.

A base apresenta características dos estudantes e suas respectivas notas nas avaliações de Matemática, Leitura e Escrita. Para realizar a leitura e manipulação dos dados, será utilizada a biblioteca **Pandas**, que permite trabalhar de forma eficiente com arquivos no formato CSV.

Após o carregamento, serão exibidas as cinco primeiras linhas do dataset para verificar se os dados foram importados corretamente e realizar uma primeira visualização de sua estrutura.

In [ ]:
# ==========================================
# 1. CARREGAMENTO DO DATASET
# ==========================================

# Importação da biblioteca Pandas
import pandas as pd

# Caminho do arquivo disponibilizado pelo Kaggle
caminho = "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"

# Carregamento do arquivo CSV
df = pd.read_csv(caminho)

# Exibição das 5 primeiras linhas do dataset
df.head()

## 2. Tradução dos Dados para Português

Para facilitar a interpretação dos dados e tornar a análise mais clara, os nomes das colunas e os valores das variáveis categóricas serão traduzidos para o português.

Essa alteração não modifica as informações originais do conjunto de dados, apenas torna sua apresentação mais adequada para a análise e para a compreensão dos resultados ao longo do projeto.

In [ ]:
# ==========================================
# 2. TRADUÇÃO DOS DADOS PARA PORTUGUÊS
# ==========================================

# Renomeando as colunas
df = df.rename(columns={
    "gender": "genero",
    "race/ethnicity": "grupo_etnico",
    "parental level of education": "escolaridade_dos_pais",
    "lunch": "tipo_de_almoco",
    "test preparation course": "curso_preparatorio",
    "math score": "nota_matematica",
    "reading score": "nota_leitura",
    "writing score": "nota_escrita"
})

# Traduzindo os valores da coluna gênero
df["genero"] = df["genero"].replace({
    "female": "feminino",
    "male": "masculino"
})

# Traduzindo a escolaridade dos pais
df["escolaridade_dos_pais"] = df["escolaridade_dos_pais"].replace({
    "bachelor's degree": "bacharelado",
    "some college": "faculdade_incompleta",
    "master's degree": "mestrado",
    "associate's degree": "curso_superior_tecnologico",
    "high school": "ensino_medio",
    "some high school": "ensino_medio_incompleto"
})

# Traduzindo o tipo de almoço
df["tipo_de_almoco"] = df["tipo_de_almoco"].replace({
    "standard": "padrao",
    "free/reduced": "gratuito_ou_reduzido"
})

# Traduzindo o curso preparatório
df["curso_preparatorio"] = df["curso_preparatorio"].replace({
    "none": "nenhum",
    "completed": "concluido"
})

# Exibindo as 5 primeiras linhas após a tradução
df.head()

## 3. Conhecendo o Dataset

Após o carregamento e a tradução dos dados, é importante conhecer a estrutura do conjunto de dados antes de iniciar o tratamento e a construção do modelo de classificação.

Nesta etapa, serão verificadas a quantidade de linhas e colunas, os nomes das variáveis e os tipos de dados presentes no dataset. Também serão apresentadas algumas informações gerais que ajudarão a compreender melhor a base utilizada no projeto.

Essa análise inicial é importante para identificar como os dados estão organizados e quais tratamentos serão necessários nas próximas etapas.

In [ ]:
# ==========================================
# 3. CONHECENDO O DATASET
# ==========================================

print("=" * 55)
print("          INFORMAÇÕES GERAIS DO DATASET")
print("=" * 55)

print(f"\nNúmero de estudantes: {df.shape[0]}")
print(f"Número de variáveis:  {df.shape[1]}")

print("\n" + "-" * 55)
print("VARIÁVEIS DISPONÍVEIS")
print("-" * 55)

for numero, coluna in enumerate(df.columns, start=1):
    
    # Identifica se a variável é numérica ou categórica
    if df[coluna].dtype == "object":
        tipo = "Categórica"
    else:
        tipo = "Numérica"
    
    print(f"{numero:02d}. {coluna:<30} | {tipo}")

print("\n" + "=" * 55)

## 4. Limpeza e Preparação dos Dados

Antes de utilizar os dados no modelo de classificação, é necessário verificar a qualidade do conjunto de dados.

Nesta etapa, serão identificados possíveis valores ausentes e registros duplicados. Esses problemas podem interferir na análise e no treinamento do modelo, por isso devem ser verificados antes das próximas etapas.

Caso sejam encontrados registros duplicados, eles serão removidos para evitar informações repetidas na base de dados.

In [ ]:
# ==========================================
# 4. LIMPEZA E PREPARAÇÃO DOS DADOS
# ==========================================

# Quantidade de valores ausentes
valores_ausentes = df.isnull().sum().sum()

# Quantidade de registros duplicados
duplicados = df.duplicated().sum()

print("=" * 50)
print("       VERIFICAÇÃO DA QUALIDADE DOS DADOS")
print("=" * 50)

print(f"\nValores ausentes encontrados: {valores_ausentes}")
print(f"Registros duplicados encontrados: {duplicados}")

# Remove registros duplicados, caso existam
if duplicados > 0:
    df = df.drop_duplicates()
    print(f"\nRegistros duplicados removidos: {duplicados}")
else:
    print("\nNenhum registro duplicado precisou ser removido.")

print(f"\nTotal de registros após a limpeza: {df.shape[0]}")

print("\n" + "=" * 50)
print("Limpeza dos dados concluída com sucesso!")
print("=" * 50)

## 5. Análise Exploratória dos Dados

Após a limpeza, será realizada uma análise exploratória para compreender melhor o desempenho dos estudantes nas três avaliações presentes no conjunto de dados: Matemática, Leitura e Escrita.

Nesta etapa, serão calculadas estatísticas como média, menor nota e maior nota de cada disciplina. Essas informações permitem observar de forma simples a distribuição geral das notas e identificar diferenças entre as avaliações antes da construção do modelo de classificação.

In [ ]:
# ==========================================
# 5. ANÁLISE EXPLORATÓRIA DAS NOTAS
# ==========================================

# Seleciona as colunas das notas
colunas_notas = [
    "nota_matematica",
    "nota_leitura",
    "nota_escrita"
]

# Calcula as estatísticas
estatisticas = df[colunas_notas].agg(["mean", "min", "max"]).T

# Renomeia as colunas
estatisticas.columns = ["Média", "Menor nota", "Maior nota"]

# Renomeia as disciplinas para apresentação
estatisticas.index = ["Matemática", "Leitura", "Escrita"]

# Arredonda a média para duas casas decimais
estatisticas["Média"] = estatisticas["Média"].round(2)

print("=" * 58)
print("             DESEMPENHO GERAL DOS ESTUDANTES")
print("=" * 58)

print(estatisticas.to_string())

print("\n" + "=" * 58)

## 6. Comparação das Médias das Notas

Para facilitar a interpretação do desempenho acadêmico dos estudantes, foi realizada uma comparação entre as médias obtidas nas avaliações de Matemática, Leitura e Escrita.

O gráfico a seguir apresenta a média de cada disciplina em uma escala de 0 a 100 pontos. Além da representação visual, cada barra apresenta o valor exato da média, permitindo identificar de maneira clara as diferenças de desempenho entre as três avaliações.

In [ ]:
# ==========================================
# 6. GRÁFICO DAS MÉDIAS DAS NOTAS
# ==========================================

import matplotlib.pyplot as plt

# Calcula a média de cada disciplina
medias = [
    df["nota_matematica"].mean(),
    df["nota_leitura"].mean(),
    df["nota_escrita"].mean()
]

disciplinas = ["Matemática", "Leitura", "Escrita"]

# Criação do gráfico
plt.figure(figsize=(8, 5))

barras = plt.bar(disciplinas, medias)

# Adiciona o valor da média acima de cada barra
for barra, media in zip(barras, medias):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 0.5,
        f"{media:.2f}",
        ha="center"
    )

plt.title("Média das Notas dos Estudantes")
plt.ylabel("Média")
plt.ylim(0, 100)

plt.tight_layout()
plt.show()

### Interpretação dos Resultados

O gráfico apresenta a média das notas dos estudantes nas três avaliações analisadas. A maior média foi observada em **Leitura, com 69,17 pontos**, seguida por **Escrita, com 68,05 pontos**, enquanto **Matemática apresentou a menor média, com 66,09 pontos**.

Apesar dessa diferença, as médias das três disciplinas estão relativamente próximas, com uma diferença de apenas **3,08 pontos** entre a maior média (Leitura) e a menor (Matemática). Isso indica que, de maneira geral, o desempenho médio dos estudantes foi semelhante nas três avaliações.

Os resultados mostram ainda que os estudantes apresentaram um desempenho médio ligeiramente superior em **Leitura e Escrita** quando comparado à Matemática. Entretanto, somente a análise das médias não é suficiente para determinar o desempenho individual dos estudantes, sendo necessário considerar a distribuição das notas e outras características presentes no conjunto de dados.

Essas informações servirão como base para as próximas etapas da análise e para a construção do modelo de classificação utilizando **Support Vector Machine (SVM)**.

## 7. Criação da Classificação de Desempenho

O conjunto de dados original apresenta as notas dos estudantes em Matemática, Leitura e Escrita, mas não possui uma variável específica indicando o nível de desempenho acadêmico.

Para possibilitar a aplicação do modelo de classificação SVM, será calculada a média individual de cada estudante considerando as três avaliações. A partir dessa média, os estudantes serão classificados em três níveis de desempenho:

- **Baixo:** média inferior a 60 pontos;
- **Médio:** média entre 60 e 79,99 pontos;
- **Alto:** média igual ou superior a 80 pontos.

Essa nova variável será denominada **desempenho** e será utilizada posteriormente como variável-alvo do modelo de classificação.

In [ ]:
# ==========================================
# 7. CRIAÇÃO DA CLASSIFICAÇÃO DE DESEMPENHO
# ==========================================

# Calcula a média individual das três avaliações
df["media_notas"] = (
    df["nota_matematica"] +
    df["nota_leitura"] +
    df["nota_escrita"]
) / 3

# Função para classificar o desempenho
def classificar_desempenho(media):
    if media < 60:
        return "Baixo"
    elif media < 80:
        return "Médio"
    else:
        return "Alto"

# Cria a nova coluna de desempenho
df["desempenho"] = df["media_notas"].apply(classificar_desempenho)

# Exibe alguns exemplos
resultado = df[
    [
        "nota_matematica",
        "nota_leitura",
        "nota_escrita",
        "media_notas",
        "desempenho"
    ]
].head(10).copy()

# Arredonda a média para facilitar a visualização
resultado["media_notas"] = resultado["media_notas"].round(2)

resultado

## 8. Distribuição dos Níveis de Desempenho

Após a criação da variável de desempenho, é importante analisar como os estudantes estão distribuídos entre as categorias **Baixo, Médio e Alto**.

Essa análise permite identificar a quantidade e a proporção de estudantes em cada nível. Além de facilitar a compreensão do perfil da base de dados, essa verificação é importante para observar se existe um desequilíbrio significativo entre as classes que serão utilizadas posteriormente no treinamento do modelo SVM.

O gráfico a seguir apresenta a quantidade de estudantes em cada categoria e seu respectivo percentual em relação ao total analisado.

In [ ]:
# ==========================================
# 8. DISTRIBUIÇÃO DOS NÍVEIS DE DESEMPENHO
# ==========================================

import matplotlib.pyplot as plt

# Define a ordem das categorias
ordem = ["Baixo", "Médio", "Alto"]

# Conta a quantidade de estudantes em cada categoria
quantidades = df["desempenho"].value_counts().reindex(ordem, fill_value=0)

# Calcula os percentuais
percentuais = (quantidades / quantidades.sum()) * 100

# Criação do gráfico
plt.figure(figsize=(9, 6))

cores = ["#E15759", "#F2CF5B", "#59A14F"]

barras = plt.bar(
    ordem,
    quantidades.values,
    color=cores,
    width=0.6,
    edgecolor="white",
    linewidth=1.5
)

# Adiciona quantidade e percentual acima das barras
for barra, quantidade, percentual in zip(
    barras,
    quantidades.values,
    percentuais.values
):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 8,
        f"{quantidade} alunos\n({percentual:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold"
    )

# Título e eixos
plt.title(
    "Distribuição dos Estudantes por Nível de Desempenho",
    fontsize=15,
    fontweight="bold",
    pad=20
)

plt.xlabel("Nível de desempenho", fontsize=11)
plt.ylabel("Quantidade de estudantes", fontsize=11)

# Grade horizontal
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

# Remove bordas superiores e laterais
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Espaço adicional para os valores acima das barras
plt.ylim(0, max(quantidades.values) * 1.20)

plt.tight_layout()
plt.show()

### Interpretação dos Resultados

O gráfico apresenta a distribuição dos **1.000 estudantes** de acordo com os três níveis de desempenho definidos a partir da média das notas de Matemática, Leitura e Escrita.

A categoria **Médio** concentra a maior parte dos estudantes, com **517 alunos (51,7%)**. Em seguida, aparecem **285 estudantes (28,5%)** classificados com desempenho **Baixo** e **198 estudantes (19,8%)** com desempenho **Alto**.

Os resultados mostram que mais da metade dos estudantes está concentrada no nível médio de desempenho. A categoria Alto possui a menor quantidade de estudantes, enquanto a categoria Baixo representa pouco mais de um quarto da base.

Também é possível observar que as três classes não possuem a mesma quantidade de exemplos. Essa diferença deverá ser considerada durante a divisão dos dados em treino e teste, para que as proporções das categorias sejam preservadas e a avaliação do modelo SVM seja mais adequada.

Essa distribuição será utilizada nas próximas etapas para preparar os dados e desenvolver o modelo responsável pela classificação do desempenho acadêmico.

## 9. Preparação e Transformação das Variáveis

Para que o algoritmo Support Vector Machine (SVM) possa ser utilizado, é necessário transformar as informações categóricas do conjunto de dados em valores numéricos.

Nesta análise, o objetivo será classificar o nível de desempenho acadêmico dos estudantes utilizando características que não correspondem diretamente às suas notas. Dessa forma, serão utilizadas como variáveis de entrada: gênero, grupo étnico, escolaridade dos pais, tipo de almoço e participação em curso preparatório.

As notas de Matemática, Leitura e Escrita, assim como a média calculada a partir delas, não serão utilizadas como variáveis de entrada do modelo, pois foram usadas para definir a própria classificação de desempenho. Essa separação evita o vazamento de dados e permite uma avaliação mais adequada da capacidade de classificação do modelo.

As variáveis categóricas serão convertidas em colunas numéricas utilizando a técnica de codificação One-Hot Encoding, tornando os dados compatíveis com o algoritmo SVM.

In [ ]:
# ==========================================
# 9. PREPARAÇÃO E TRANSFORMAÇÃO DAS VARIÁVEIS
# ==========================================

# Seleciona somente as características que serão utilizadas pelo modelo
X = df[
    [
        "genero",
        "grupo_etnico",
        "escolaridade_dos_pais",
        "tipo_de_almoco",
        "curso_preparatorio"
    ]
].copy()

# Define a variável que o modelo deverá classificar
y = df["desempenho"].copy()

# Transforma as variáveis categóricas em variáveis numéricas
X = pd.get_dummies(X, drop_first=True, dtype=int)

# Exibe um resumo da transformação
print("=" * 58)
print("          PREPARAÇÃO DAS VARIÁVEIS PARA O SVM")
print("=" * 58)

print(f"\nQuantidade de estudantes: {X.shape[0]}")
print(f"Variáveis após a transformação: {X.shape[1]}")

print("\nVariável que será prevista:")
print("→ desempenho (Baixo, Médio ou Alto)")

print("\nCaracterísticas utilizadas:")
print("→ Gênero")
print("→ Grupo étnico")
print("→ Escolaridade dos pais")
print("→ Tipo de almoço")
print("→ Curso preparatório")

print("\n" + "=" * 58)
print("Transformação concluída com sucesso!")
print("=" * 58)

# Mostra as primeiras linhas dos dados transformados
X.head()

## 10. Separação dos Dados em Treino e Teste

Após a transformação das variáveis categóricas, o conjunto de dados será dividido em duas partes: **treinamento e teste**.

O conjunto de treinamento será utilizado para que o modelo SVM aprenda os padrões presentes nos dados, enquanto o conjunto de teste será utilizado posteriormente para avaliar o desempenho do modelo em dados que não foram utilizados durante o treinamento.

Neste projeto, serão utilizados **80% dos dados para treinamento e 20% para teste**. A divisão será realizada de forma estratificada, preservando aproximadamente a proporção de estudantes classificados nos níveis Baixo, Médio e Alto em ambos os conjuntos.

Também será definida uma semente aleatória para que a divisão possa ser reproduzida da mesma forma em novas execuções do Notebook.

In [ ]:
# ==========================================
# 10. SEPARAÇÃO DOS DADOS EM TREINO E TESTE
# ==========================================

from sklearn.model_selection import train_test_split

# Divide os dados em 80% para treinamento e 20% para teste
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Exibe as informações da divisão
print("=" * 58)
print("          DIVISÃO DOS DADOS EM TREINO E TESTE")
print("=" * 58)

print(f"\nTotal de estudantes: {len(X)}")
print(f"Dados para treinamento: {len(X_treino)} ({len(X_treino)/len(X)*100:.0f}%)")
print(f"Dados para teste:       {len(X_teste)} ({len(X_teste)/len(X)*100:.0f}%)")

print("\nDistribuição das classes no treinamento:")
for classe, quantidade in y_treino.value_counts().reindex(
    ["Baixo", "Médio", "Alto"]
).items():
    percentual = quantidade / len(y_treino) * 100
    print(f"→ {classe:<5}: {quantidade:3d} estudantes ({percentual:.1f}%)")

print("\nDistribuição das classes no teste:")
for classe, quantidade in y_teste.value_counts().reindex(
    ["Baixo", "Médio", "Alto"]
).items():
    percentual = quantidade / len(y_teste) * 100
    print(f"→ {classe:<5}: {quantidade:3d} estudantes ({percentual:.1f}%)")

print("\n" + "=" * 58)
print("Divisão concluída com sucesso!")
print("=" * 58)

## 11. Padronização dos Dados

Antes do treinamento do modelo SVM, será realizada a padronização das variáveis de entrada utilizando o **StandardScaler**.

A padronização transforma os dados para que as variáveis fiquem em uma escala comparável, evitando que diferenças de escala influenciem de maneira inadequada o processo de treinamento do modelo.

O `StandardScaler` será ajustado utilizando somente os dados de treinamento. Em seguida, a mesma transformação será aplicada aos dados de teste. Esse procedimento evita que informações do conjunto de teste sejam utilizadas durante a preparação do modelo.

In [ ]:
# ==========================================
# 11. PADRONIZAÇÃO DOS DADOS
# ==========================================

from sklearn.preprocessing import StandardScaler

# Criação do padronizador
scaler = StandardScaler()

# Ajusta o padronizador e transforma os dados de treinamento
X_treino_padronizado = scaler.fit_transform(X_treino)

# Aplica a mesma transformação aos dados de teste
X_teste_padronizado = scaler.transform(X_teste)

print("=" * 58)
print("              PADRONIZAÇÃO DOS DADOS")
print("=" * 58)

print(f"\nDados de treinamento: {X_treino_padronizado.shape[0]} estudantes")
print(f"Dados de teste:       {X_teste_padronizado.shape[0]} estudantes")
print(f"Variáveis utilizadas: {X_treino_padronizado.shape[1]}")

print("\nO StandardScaler foi ajustado somente com")
print("os dados do conjunto de treinamento.")

print("\n" + "=" * 58)
print("Padronização concluída com sucesso!")
print("=" * 58)

## 12. Treinamento dos Modelos SVM

Após a preparação, divisão e padronização dos dados, será realizado o treinamento dos modelos utilizando o algoritmo **Support Vector Machine (SVM)**.

O SVM é um algoritmo de aprendizado supervisionado utilizado em problemas de classificação. Seu objetivo é encontrar limites de decisão capazes de separar as diferentes classes presentes nos dados.

Neste projeto, serão testados dois tipos de kernel:

- **Linear:** busca separar as classes utilizando limites de decisão lineares;
- **RBF (Radial Basis Function):** permite identificar relações não lineares e padrões mais complexos entre as características.

Os dois modelos serão treinados utilizando o mesmo conjunto de treinamento padronizado. Posteriormente, seus resultados serão comparados para analisar o comportamento de cada configuração na classificação dos níveis de desempenho acadêmico.

In [ ]:
# ==========================================
# 12. TREINAMENTO DOS MODELOS SVM
# ==========================================

from sklearn.svm import SVC

# Criação do modelo SVM com kernel Linear
svm_linear = SVC(
    kernel="linear",
    C=1.0
)

# Criação do modelo SVM com kernel RBF
svm_rbf = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale"
)

# Treinamento do modelo Linear
svm_linear.fit(
    X_treino_padronizado,
    y_treino
)

# Treinamento do modelo RBF
svm_rbf.fit(
    X_treino_padronizado,
    y_treino
)

print("=" * 58)
print("             TREINAMENTO DOS MODELOS SVM")
print("=" * 58)

print("\nModelo 1")
print("→ Kernel: Linear")
print("→ C: 1.0")
print("→ Status: Treinado com sucesso")

print("\nModelo 2")
print("→ Kernel: RBF")
print("→ C: 1.0")
print("→ Gamma: Scale")
print("→ Status: Treinado com sucesso")

print(f"\nQuantidade de dados utilizados no treinamento: {len(y_treino)}")

print("\n" + "=" * 58)
print("Treinamento dos modelos concluído!")
print("=" * 58)

## 13. Previsões e Avaliação da Acurácia

Após o treinamento dos modelos SVM, será realizada a etapa de previsão utilizando o conjunto de teste, composto por dados que não foram utilizados durante o treinamento.

Os modelos com kernel **Linear** e **RBF** serão utilizados para classificar os estudantes nos níveis de desempenho Baixo, Médio ou Alto.

Para realizar uma primeira avaliação dos modelos, será utilizada a **acurácia**, que representa a proporção de classificações corretas em relação ao total de exemplos analisados.

A comparação entre os dois modelos permitirá observar como cada tipo de kernel se comportou diante dos dados utilizados neste projeto.

In [ ]:
# ==========================================
# 13. PREVISÕES E AVALIAÇÃO DA ACURÁCIA
# ==========================================

from sklearn.metrics import accuracy_score

# Realiza as previsões com os dados de teste
previsao_linear = svm_linear.predict(X_teste_padronizado)
previsao_rbf = svm_rbf.predict(X_teste_padronizado)

# Calcula a acurácia dos modelos
acuracia_linear = accuracy_score(y_teste, previsao_linear)
acuracia_rbf = accuracy_score(y_teste, previsao_rbf)

# Calcula a quantidade de acertos
acertos_linear = (y_teste == previsao_linear).sum()
acertos_rbf = (y_teste == previsao_rbf).sum()

# Total de estudantes utilizados no teste
total_teste = len(y_teste)

print("=" * 60)
print("             RESULTADOS DOS MODELOS SVM")
print("=" * 60)

print("\nSVM - KERNEL LINEAR")
print(f"→ Acertos:   {acertos_linear} de {total_teste}")
print(f"→ Erros:     {total_teste - acertos_linear} de {total_teste}")
print(f"→ Acurácia:  {acuracia_linear * 100:.2f}%")

print("\nSVM - KERNEL RBF")
print(f"→ Acertos:   {acertos_rbf} de {total_teste}")
print(f"→ Erros:     {total_teste - acertos_rbf} de {total_teste}")
print(f"→ Acurácia:  {acuracia_rbf * 100:.2f}%")

print("\n" + "=" * 60)
print("Avaliação inicial concluída!")
print("=" * 60)

## 14. Matrizes de Confusão

Após a análise da acurácia, serão utilizadas matrizes de confusão para avaliar de forma mais detalhada o comportamento dos modelos SVM.

A matriz de confusão permite comparar as classificações reais com as previsões realizadas pelo modelo, mostrando quantos estudantes foram classificados corretamente e quais categorias foram confundidas.

Neste projeto, serão analisadas separadamente as matrizes dos modelos com kernel **Linear** e **RBF**, considerando as três categorias de desempenho: Baixo, Médio e Alto.

Os valores localizados na diagonal principal representam as classificações corretas, enquanto os demais valores representam os erros cometidos pelo modelo.

In [ ]:
# ==========================================
# 14. MATRIZES DE CONFUSÃO
# ==========================================

import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Ordem das classes
classes = ["Baixo", "Médio", "Alto"]

# Calcula as matrizes de confusão
matriz_linear = confusion_matrix(
    y_teste,
    previsao_linear,
    labels=classes
)

matriz_rbf = confusion_matrix(
    y_teste,
    previsao_rbf,
    labels=classes
)

# ------------------------------------------
# MATRIZ DE CONFUSÃO - SVM LINEAR
# ------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

display_linear = ConfusionMatrixDisplay(
    confusion_matrix=matriz_linear,
    display_labels=classes
)

display_linear.plot(
    ax=ax,
    cmap="Blues",
    colorbar=False
)

plt.title(
    "Matriz de Confusão - SVM Linear",
    fontsize=15,
    fontweight="bold",
    pad=15
)

plt.xlabel("Classificação prevista", fontsize=11)
plt.ylabel("Classificação real", fontsize=11)

plt.tight_layout()
plt.show()


# ------------------------------------------
# MATRIZ DE CONFUSÃO - SVM RBF
# ------------------------------------------

fig, ax = plt.subplots(figsize=(7, 6))

display_rbf = ConfusionMatrixDisplay(
    confusion_matrix=matriz_rbf,
    display_labels=classes
)

display_rbf.plot(
    ax=ax,
    cmap="Greens",
    colorbar=False
)

plt.title(
    "Matriz de Confusão - SVM RBF",
    fontsize=15,
    fontweight="bold",
    pad=15
)

plt.xlabel("Classificação prevista", fontsize=11)
plt.ylabel("Classificação real", fontsize=11)

plt.tight_layout()
plt.show()

### Interpretação das Matrizes de Confusão

As matrizes de confusão permitem observar de forma detalhada o comportamento dos modelos SVM Linear e RBF na classificação dos níveis de desempenho acadêmico.

No modelo **SVM Linear**, dos 57 estudantes com desempenho Baixo, 20 foram classificados corretamente, enquanto 37 foram classificados como Médio. Entre os 103 estudantes da categoria Médio, o modelo classificou corretamente 93 e classificou 10 como Baixo. Já entre os 40 estudantes com desempenho Alto, nenhum foi classificado corretamente, sendo a maioria classificada como Médio.

O modelo **SVM RBF** apresentou comportamento semelhante. Dos 57 estudantes da categoria Baixo, 17 foram classificados corretamente e 40 foram classificados como Médio. Na categoria Médio, 90 dos 103 estudantes foram classificados corretamente. Novamente, nenhum dos 40 estudantes da categoria Alto foi identificado corretamente pelo modelo.

Os resultados mostram que ambos os modelos apresentam uma forte tendência de classificar os estudantes na categoria **Médio**. Essa categoria também é a mais frequente no conjunto de dados, representando 51,7% dos estudantes.

A dificuldade de classificação das categorias Baixo e Alto indica que as características utilizadas como entrada — gênero, grupo étnico, escolaridade dos pais, tipo de almoço e participação em curso preparatório — não foram suficientes para separar claramente os três níveis de desempenho.

É importante destacar que as notas de Matemática, Leitura e Escrita não foram utilizadas como características de entrada, pois elas foram utilizadas para construir a própria variável de desempenho. Sua utilização no treinamento provocaria vazamento de dados e poderia produzir resultados artificialmente elevados.

Portanto, apesar de os modelos conseguirem identificar uma parcela considerável dos estudantes da categoria Médio, os resultados demonstram limitações importantes na classificação das demais categorias.

## 15. Relatório de Classificação

Além da acurácia e da matriz de confusão, serão utilizadas outras métricas para avaliar de forma mais detalhada o desempenho dos modelos SVM.

As métricas analisadas serão:

- **Precisão (Precision):** indica, entre as previsões realizadas para determinada classe, quantas estavam corretas.
- **Recall:** indica a capacidade do modelo de identificar corretamente os exemplos que realmente pertencem a determinada classe.
- **F1-score:** representa um equilíbrio entre precisão e recall.
- **Support:** representa a quantidade de estudantes de cada classe presente no conjunto de teste.

Essas métricas são importantes porque permitem analisar separadamente o desempenho dos modelos nas categorias Baixo, Médio e Alto, fornecendo uma avaliação mais completa do que a utilização isolada da acurácia.

In [ ]:
# ==========================================
# 15. RELATÓRIO DE CLASSIFICAÇÃO
# ==========================================

from sklearn.metrics import classification_report

# Ordem das classes
classes = ["Baixo", "Médio", "Alto"]

print("=" * 65)
print("            RELATÓRIO DE CLASSIFICAÇÃO - SVM LINEAR")
print("=" * 65)

print(
    classification_report(
        y_teste,
        previsao_linear,
        labels=classes,
        target_names=classes,
        digits=2,
        zero_division=0
    )
)

print("=" * 65)
print("              RELATÓRIO DE CLASSIFICAÇÃO - SVM RBF")
print("=" * 65)

print(
    classification_report(
        y_teste,
        previsao_rbf,
        labels=classes,
        target_names=classes,
        digits=2,
        zero_division=0
    )
)

## 16. Comparação da Acurácia dos Modelos SVM

Após a avaliação individual dos modelos, será realizada uma comparação visual entre as acurácias obtidas pelos modelos SVM com kernel Linear e RBF.

A acurácia representa a proporção de classificações corretas realizadas pelo modelo em relação ao total de estudantes presentes no conjunto de teste.

O gráfico a seguir permite visualizar de maneira simples as diferenças de desempenho entre os dois kernels utilizados. Entretanto, essa métrica deve ser interpretada em conjunto com as matrizes de confusão e o relatório de classificação, pois a acurácia isoladamente não demonstra o desempenho do modelo em cada uma das três classes.

In [ ]:
# ==========================================
# 16. COMPARAÇÃO DA ACURÁCIA DOS MODELOS
# ==========================================

import matplotlib.pyplot as plt

# Nomes dos modelos
modelos = ["SVM Linear", "SVM RBF"]

# Converte as acurácias para porcentagem
acuracias = [
    acuracia_linear * 100,
    acuracia_rbf * 100
]

# Criação do gráfico
plt.figure(figsize=(9, 6))

cores = ["#4C78A8", "#59A14F"]

barras = plt.bar(
    modelos,
    acuracias,
    color=cores,
    width=0.55,
    edgecolor="white",
    linewidth=1.5
)

# Exibe o valor exato sobre cada barra
for barra, acuracia in zip(barras, acuracias):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 1.5,
        f"{acuracia:.2f}%",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

# Título e identificação dos eixos
plt.title(
    "Comparação da Acurácia dos Modelos SVM",
    fontsize=15,
    fontweight="bold",
    pad=20
)

plt.ylabel("Acurácia (%)", fontsize=11)
plt.xlabel("Modelo utilizado", fontsize=11)

# Escala percentual
plt.ylim(0, 100)

# Linhas horizontais para facilitar a leitura
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

# Remove bordas desnecessárias
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### Interpretação dos Resultados

O gráfico apresenta a comparação da acurácia obtida pelos dois modelos SVM avaliados. O modelo com **kernel Linear alcançou acurácia de 56,50%**, enquanto o modelo com **kernel RBF apresentou acurácia de 53,50%**.

Considerando os 200 estudantes utilizados no conjunto de teste, esses percentuais correspondem a **113 classificações corretas no modelo Linear** e **107 classificações corretas no modelo RBF**. Portanto, o SVM Linear apresentou uma acurácia 3 pontos percentuais superior à obtida pelo SVM RBF nesta configuração.

Entretanto, a acurácia não deve ser analisada isoladamente. As matrizes de confusão mostraram que ambos os modelos apresentaram maior facilidade para classificar estudantes da categoria **Médio** e dificuldade principalmente na identificação da categoria **Alto**.

Esse comportamento indica que as características utilizadas como entrada no modelo não fornecem uma separação clara entre os três níveis de desempenho. Dessa forma, a avaliação completa deve considerar também métricas como precisão, recall e F1-score, além da matriz de confusão.

Os resultados obtidos até esta etapa demonstram que a alteração do kernel de Linear para RBF, utilizando os parâmetros inicialmente definidos, não proporcionou aumento na acurácia do modelo.

## 17. Ajuste de Hiperparâmetros do SVM

Após a avaliação inicial dos modelos SVM, será realizado um ajuste de hiperparâmetros com o objetivo de encontrar configurações mais adequadas para os dados analisados.

O parâmetro **C** controla o equilíbrio entre a busca por uma separação mais rígida das classes e a tolerância a erros durante o treinamento. Para o kernel RBF, também será analisado o parâmetro **gamma**, responsável por controlar a influência das observações na construção dos limites de decisão.

Para realizar essa busca será utilizado o **GridSearchCV**, que testa diferentes combinações de parâmetros utilizando validação cruzada. Neste projeto, será utilizada validação cruzada estratificada com 5 divisões, preservando a proporção das classes.

A seleção será realizada considerando o **F1-score macro**, que atribui a mesma importância às categorias Baixo, Médio e Alto. Essa escolha é adequada porque as classes possuem quantidades diferentes de estudantes e a acurácia isolada pode favorecer a categoria mais frequente.

In [ ]:
# ==========================================
# 17. AJUSTE DE HIPERPARÂMETROS DO SVM
# ==========================================

from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Parâmetros que serão testados
parametros = [
    {
        "kernel": ["linear"],
        "C": [0.1, 1, 10, 100]
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10, 100],
        "gamma": ["scale", 0.01, 0.1, 1]
    }
]

# Criação da busca
busca_svm = GridSearchCV(
    estimator=SVC(),
    param_grid=parametros,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

# Treinamento e busca das melhores configurações
busca_svm.fit(
    X_treino_padronizado,
    y_treino
)

# Melhor modelo encontrado
melhor_svm = busca_svm.best_estimator_

print("=" * 60)
print("           AJUSTE DE HIPERPARÂMETROS DO SVM")
print("=" * 60)

print("\nMelhores parâmetros encontrados:")

for parametro, valor in busca_svm.best_params_.items():
    print(f"→ {parametro}: {valor}")

print(
    f"\nMelhor F1-score médio na validação cruzada: "
    f"{busca_svm.best_score_:.4f}"
)

print("\n" + "=" * 60)
print("Busca de hiperparâmetros concluída!")
print("=" * 60)

## 18. Avaliação do Modelo SVM Otimizado

Após a busca pelos melhores hiperparâmetros, o modelo selecionado será avaliado utilizando o conjunto de teste.

Esse conjunto contém 200 estudantes que não foram utilizados na escolha dos hiperparâmetros, permitindo avaliar o comportamento do modelo em dados separados do processo de treinamento e otimização.

Serão calculadas a acurácia e a quantidade de classificações corretas e incorretas. Além disso, o resultado será comparado com os modelos SVM Linear e RBF utilizados inicialmente.

In [ ]:
# ==========================================
# 18. AVALIAÇÃO DO MODELO SVM OTIMIZADO
# ==========================================

from sklearn.metrics import accuracy_score

# Realiza as previsões com o melhor modelo
previsao_otimizada = melhor_svm.predict(X_teste_padronizado)

# Calcula a acurácia
acuracia_otimizada = accuracy_score(
    y_teste,
    previsao_otimizada
)

# Quantidade de acertos e erros
acertos_otimizado = (y_teste == previsao_otimizada).sum()
erros_otimizado = len(y_teste) - acertos_otimizado

print("=" * 62)
print("             RESULTADO DO SVM OTIMIZADO")
print("=" * 62)

print(f"\nEstudantes avaliados: {len(y_teste)}")
print(f"→ Acertos:   {acertos_otimizado}")
print(f"→ Erros:     {erros_otimizado}")
print(f"→ Acurácia:  {acuracia_otimizada * 100:.2f}%")

print("\nMelhores parâmetros utilizados:")

for parametro, valor in busca_svm.best_params_.items():
    print(f"→ {parametro}: {valor}")

print("\n" + "-" * 62)
print("COMPARAÇÃO DAS ACURÁCIAS")
print("-" * 62)

print(f"SVM Linear inicial: {acuracia_linear * 100:.2f}%")
print(f"SVM RBF inicial:    {acuracia_rbf * 100:.2f}%")
print(f"SVM Otimizado:      {acuracia_otimizada * 100:.2f}%")

print("\n" + "=" * 62)
print("Avaliação do modelo otimizado concluída!")
print("=" * 62)

## 19. Avaliação Detalhada do Modelo SVM Otimizado

Após a avaliação da acurácia do modelo otimizado, será realizada uma análise mais detalhada de seu desempenho.

Para isso, serão utilizadas a **matriz de confusão** e as métricas de **precisão, recall e F1-score**. Essas informações permitem verificar não apenas a quantidade total de classificações corretas, mas também a capacidade do modelo de identificar individualmente os estudantes das categorias Baixo, Médio e Alto.

Essa análise é especialmente importante neste projeto, pois os modelos iniciais apresentaram maior facilidade para identificar a categoria Médio e dificuldade na classificação das demais categorias.

In [ ]:
# ==========================================
# 19. AVALIAÇÃO DETALHADA DO SVM OTIMIZADO
# ==========================================

import matplotlib.pyplot as plt

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

# Ordem das classes
classes = ["Baixo", "Médio", "Alto"]

# Calcula a matriz de confusão
matriz_otimizada = confusion_matrix(
    y_teste,
    previsao_otimizada,
    labels=classes
)

# Criação do gráfico
fig, ax = plt.subplots(figsize=(7, 6))

display_otimizado = ConfusionMatrixDisplay(
    confusion_matrix=matriz_otimizada,
    display_labels=classes
)

display_otimizado.plot(
    ax=ax,
    cmap="Purples",
    colorbar=False
)

plt.title(
    "Matriz de Confusão - SVM Otimizado",
    fontsize=15,
    fontweight="bold",
    pad=15
)

plt.xlabel("Classificação prevista", fontsize=11)
plt.ylabel("Classificação real", fontsize=11)

plt.tight_layout()
plt.show()


# Relatório de classificação
print("=" * 65)
print("        RELATÓRIO DE CLASSIFICAÇÃO - SVM OTIMIZADO")
print("=" * 65)

print(
    classification_report(
        y_teste,
        previsao_otimizada,
        labels=classes,
        target_names=classes,
        digits=2,
        zero_division=0
    )
)

### Interpretação dos Resultados do SVM Otimizado

A matriz de confusão apresenta o desempenho do modelo SVM após o processo de ajuste dos hiperparâmetros.

Na categoria **Baixo**, dos 57 estudantes presentes no conjunto de teste, **20 foram classificados corretamente**. Entretanto, 36 foram classificados como Médio e 1 como Alto.

Na categoria **Médio**, que possui 103 estudantes, o modelo apresentou seu melhor desempenho, classificando corretamente **84 estudantes**. Os demais foram classificados incorretamente, sendo 17 como Baixo e 2 como Alto.

Na categoria **Alto**, dos 40 estudantes existentes no conjunto de teste, apenas **4 foram identificados corretamente**. Outros 33 foram classificados como Médio e 3 como Baixo.

No total, o modelo realizou **108 classificações corretas entre os 200 estudantes**, correspondendo a uma acurácia de aproximadamente **54%**.

Comparando esse resultado com os modelos iniciais, observa-se que o modelo otimizado passou a identificar alguns estudantes da categoria Alto, enquanto os modelos Linear e RBF iniciais não haviam realizado nenhuma classificação correta nessa categoria. Entretanto, essa melhoria ocorreu acompanhada de uma redução na quantidade total de classificações corretas em relação ao SVM Linear inicial, que apresentou acurácia de 56,50%.

A matriz também demonstra que o modelo continua apresentando uma tendência de classificar grande parte dos estudantes como pertencentes à categoria **Médio**. Isso indica que as características utilizadas como entrada possuem capacidade limitada para distinguir claramente os três níveis de desempenho acadêmico.

Dessa forma, o ajuste dos hiperparâmetros alterou o comportamento do modelo e permitiu alguma identificação da categoria Alto, mas não solucionou completamente a dificuldade de separação entre as três classes.

## 20. Comparação Final dos Modelos SVM

Após o treinamento, avaliação e otimização dos modelos, será realizada uma comparação final entre as três configurações utilizadas: **SVM Linear, SVM RBF e SVM Otimizado**.

Essa comparação permite visualizar as diferenças de acurácia entre os modelos e analisar o efeito do ajuste de hiperparâmetros.

É importante destacar que a escolha e a análise de um modelo de classificação não devem considerar apenas a acurácia. As matrizes de confusão e as métricas por classe também são importantes, principalmente neste projeto, devido à diferença na quantidade de estudantes entre as categorias Baixo, Médio e Alto.

In [ ]:
# ==========================================
# 20. COMPARAÇÃO FINAL DOS MODELOS SVM
# ==========================================

import pandas as pd
import matplotlib.pyplot as plt

# Organiza os resultados dos três modelos
comparacao = pd.DataFrame({
    "Modelo": [
        "SVM Linear",
        "SVM RBF",
        "SVM Otimizado"
    ],
    "Acurácia (%)": [
        acuracia_linear * 100,
        acuracia_rbf * 100,
        acuracia_otimizada * 100
    ]
})

# Arredonda os resultados
comparacao["Acurácia (%)"] = comparacao["Acurácia (%)"].round(2)

print("=" * 55)
print("          COMPARAÇÃO FINAL DOS MODELOS")
print("=" * 55)

print(comparacao.to_string(index=False))

print("\n" + "=" * 55)


# ==========================================
# GRÁFICO COMPARATIVO
# ==========================================

plt.figure(figsize=(10, 6))

cores = ["#4C78A8", "#59A14F", "#8064A2"]

barras = plt.bar(
    comparacao["Modelo"],
    comparacao["Acurácia (%)"],
    color=cores,
    width=0.6,
    edgecolor="white",
    linewidth=1.5
)

# Mostra a acurácia acima de cada barra
for barra, valor in zip(barras, comparacao["Acurácia (%)"]):
    plt.text(
        barra.get_x() + barra.get_width() / 2,
        barra.get_height() + 1,
        f"{valor:.2f}%",
        ha="center",
        fontsize=12,
        fontweight="bold"
    )

plt.title(
    "Comparação Final da Acurácia dos Modelos SVM",
    fontsize=15,
    fontweight="bold",
    pad=20
)

plt.xlabel("Modelo")
plt.ylabel("Acurácia (%)")

plt.ylim(0, 100)

plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.25
)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### Interpretação da Comparação Final

A comparação mostra que os três modelos apresentaram resultados relativamente próximos. O **SVM Linear obteve a maior acurácia geral, com 56,50%**, seguido pelo modelo SVM otimizado, com aproximadamente **54,00%**, e pelo SVM RBF inicial, com **53,50%**.

Embora o modelo Linear tenha apresentado a maior quantidade geral de classificações corretas, as análises anteriores mostraram que ele não conseguiu identificar corretamente estudantes da categoria Alto.

O modelo otimizado apresentou acurácia geral inferior à do modelo Linear, porém conseguiu classificar corretamente alguns estudantes da categoria Alto, demonstrando uma mudança na distribuição das previsões após o ajuste dos hiperparâmetros.

Portanto, os resultados demonstram que a acurácia isoladamente não é suficiente para avaliar completamente os modelos. As matrizes de confusão e as métricas de classificação devem ser consideradas em conjunto para compreender o comportamento dos modelos em cada categoria.

De maneira geral, os resultados também indicam que as características disponíveis como entrada possuem capacidade limitada para distinguir os níveis de desempenho acadêmico definidos neste estudo.

## 21. Conclusão

Este projeto teve como objetivo desenvolver e avaliar modelos de classificação utilizando **Support Vector Machine (SVM)** para analisar o desempenho acadêmico de estudantes a partir do conjunto de dados **Students Performance in Exams**, disponibilizado na plataforma Kaggle.

Inicialmente, foi realizada a preparação do conjunto de dados, incluindo o carregamento, tradução das variáveis, análise da estrutura, verificação de valores ausentes e registros duplicados. Em seguida, foram analisadas as notas de Matemática, Leitura e Escrita e criada uma variável de desempenho acadêmico dividida em três categorias: **Baixo, Médio e Alto**.

A distribuição dos estudantes mostrou que **51,7% pertenciam à categoria Médio, 28,5% à categoria Baixo e 19,8% à categoria Alto**. Por esse motivo, a divisão entre treinamento e teste foi realizada de forma estratificada, preservando aproximadamente essas proporções nos dois conjuntos.

Para evitar vazamento de dados, as notas e a média utilizadas na criação da variável de desempenho não foram fornecidas como características de entrada para o SVM. Dessa forma, os modelos utilizaram informações como gênero, grupo étnico, escolaridade dos pais, tipo de almoço e participação em curso preparatório.

Foram inicialmente avaliados dois modelos SVM. O modelo com **kernel Linear apresentou acurácia de 56,50%**, enquanto o modelo com **kernel RBF alcançou 53,50%**. As matrizes de confusão mostraram que ambos apresentaram maior facilidade para identificar estudantes da categoria Médio e dificuldade principalmente na classificação da categoria Alto.

Posteriormente, foi realizado um ajuste de hiperparâmetros utilizando **GridSearchCV e validação cruzada**, considerando o F1-score macro como métrica de seleção. O modelo otimizado apresentou acurácia de aproximadamente **54%** e conseguiu identificar corretamente 4 estudantes da categoria Alto, enquanto os modelos iniciais não haviam identificado corretamente nenhum estudante dessa categoria.

Os resultados demonstram que a acurácia não deve ser utilizada isoladamente para avaliar modelos de classificação, especialmente quando existe diferença na quantidade de exemplos entre as classes. Métricas como precisão, recall, F1-score e a matriz de confusão foram fundamentais para compreender o comportamento dos modelos em cada categoria.

De maneira geral, os experimentos indicaram que as características disponíveis como entrada possuem **capacidade limitada para distinguir os três níveis de desempenho acadêmico**. Isso não significa que o algoritmo SVM não seja adequado, mas evidencia que a qualidade da classificação também depende das informações disponíveis para o modelo.

Como possibilidade de trabalhos futuros, poderiam ser utilizados conjuntos de dados com outras características relacionadas ao contexto acadêmico, como frequência, horas de estudo, participação nas aulas e histórico escolar. Essas informações poderiam fornecer novos elementos para investigar e potencialmente melhorar a classificação do desempenho dos estudantes.